# classify scenes on cloud cover over lake
Unfortunately, I am not completely sure how to use the `QADataPlane` and `QADataPlane2` data provided along with the surface kinetic energy in the ASTER08 004 data. The cloud mask provided in `QADataPlane` tif encoded as int value 0 does not seem to mask clouds reliably (although it is supposed to encode best data quality, see https://asterweb.jpl.nasa.gov/content/03_data/04_Documents/ASTER%20QA%20Plan%20v2.0.pdf, page 31), as inspection has indicated. Indeed, most scenes do not contain any cloud classified pixels. 

Since there are only roughly 200 scenes for 2000-2026 for lake Kangaarsuup Tasersua (lake id 6 in dataset by Penny), I can also do this in a quick and reliable way manually, although I am aware that this approach cannot be scaled up easily, in case the analysis is supposed to get extended to more lakes. (The ASTER08 would be great for this purpose, since it hat a spatial resolution of 90m!, however, the temporal resolution is not that great, roughly every 14days)

This script creates an image with the plotted scene and the lake boundary, and creates some buttons, which can be selected to classify the degree of cloud cover (`clear`, `partally covered` `cloudy` and `outside_of_scene`), and dependent on what is clicked, the according information is written into a `.csv` file together with the filename, which can later be used to use only the scenes where no clouds are present). 


IMPORTANT: This needs to be run in a jupyter notebook in order to work, I was not able to use it in a positron python interpreter (similar to vs code). This means, that also jupyter needs to be installed in the conda environment (called `griml`)

In [6]:
import pandas as pd
import numpy as np
import os
import rasterio
import geopandas as gpd
import xarray as xr
import rioxarray 
import pathlib
import matplotlib.pyplot as plt

In [2]:
%matplotlib widget

In [3]:
# read files 
folder = pathlib.Path(r"C:\Users\Lenovo\Documents\GEUS_internship\LWST_estimation_project\ASTER08_LSWT\data")
files = sorted(folder.glob("*_SKT.tif"))
files_QA = sorted(folder.glob("*_QA_DataPlane.tif"))
files_QA2 = sorted(folder.glob("*_QA_DataPlane.tif"))

print(files)

lake = gpd.read_file("lake_shoreline.shp")
lake6_shore_32622 = lake.to_crs(epsg = 32622)

[WindowsPath('C:/Users/Lenovo/Documents/GEUS_internship/LWST_estimation_project/ASTER08_LSWT/data/AST_08_00403082008145510_20250620191042_SKT.tif'), WindowsPath('C:/Users/Lenovo/Documents/GEUS_internship/LWST_estimation_project/ASTER08_LSWT/data/AST_08_00403082008145519_20250620191317_SKT.tif'), WindowsPath('C:/Users/Lenovo/Documents/GEUS_internship/LWST_estimation_project/ASTER08_LSWT/data/AST_08_00403152008150116_20250620211308_SKT.tif'), WindowsPath('C:/Users/Lenovo/Documents/GEUS_internship/LWST_estimation_project/ASTER08_LSWT/data/AST_08_00404122003145553_20250308145600_SKT.tif'), WindowsPath('C:/Users/Lenovo/Documents/GEUS_internship/LWST_estimation_project/ASTER08_LSWT/data/AST_08_00404122003145602_20250308145051_SKT.tif'), WindowsPath('C:/Users/Lenovo/Documents/GEUS_internship/LWST_estimation_project/ASTER08_LSWT/data/AST_08_00404212003144939_20250308192344_SKT.tif'), WindowsPath('C:/Users/Lenovo/Documents/GEUS_internship/LWST_estimation_project/ASTER08_LSWT/data/AST_08_0040421

In [4]:
img = rioxarray.open_rasterio(files[0])

print("nodata:", img.rio.nodata)
print("min:", float(img.min()))
print("max:", float(img.max()))

print(
    "percentiles:",
    img.quantile([0, 0.01, 0.05, 0.5, 0.95, 0.99, 1]).values
)

nodata: None
min: 2000.0
max: 3563.0
percentiles: [2000. 2000. 2000. 2432. 2484. 2530. 3563.]


In [5]:
import os
import io
import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display
from rasterio.plot import plotting_extent


# =========================================================
# SETTINGS
# =========================================================

LABEL_FILE = "cloud_labels.csv"


# =========================================================
# LOAD EXISTING LABELS
# =========================================================

if os.path.exists(LABEL_FILE) and os.path.getsize(LABEL_FILE) > 0:

    try:
        labels_df = pd.read_csv(LABEL_FILE)

        if "file" in labels_df.columns and "label" in labels_df.columns:
            labels = dict(
                zip(
                    labels_df["file"].astype(str),
                    labels_df["label"]
                )
            )
        else:
            labels = {}

    except pd.errors.EmptyDataError:
        labels = {}

else:
    labels = {}

print(f"Already labelled: {len(labels)}")


# =========================================================
# IMAGE FUNCTION
# =========================================================

def make_image_png(file):

    # -----------------------------------------------------
    # OPEN ASTER RASTER WITH RIOXARRAY
    # -----------------------------------------------------

    img = rioxarray.open_rasterio(file)
    
    # Mask nodata values
    img = img.where(img != img.rio.nodata)
    img = img.where(img != 2000)


    # ASTER temperature scaling
    da = (img * 0.1) - 273.15

    # -----------------------------------------------------
    # CREATE FIGURE
    # -----------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(12, 8),
        dpi=120
    )

    # -----------------------------------------------------
    # TEMPERATURE RASTER
    # -----------------------------------------------------

    da.plot(
        ax=ax,
        x="xc",
        y="yc",
        cmap="inferno",
        #vmin=-30,
        #vmax=30,
        add_colorbar=True,
        cbar_kwargs={
            "label": "Surface temperature (°C)"
        }
    )

    # -----------------------------------------------------
    # LAKE SHORELINE
    # -----------------------------------------------------

    lake6_shore_32622.plot(
        ax=ax,
        facecolor="none",
        edgecolor="cyan",
        linewidth=2
    )

    # -----------------------------------------------------
    # AXES / TITLE
    # -----------------------------------------------------

    ax.set_xlabel("UTM Easting (m)")
    ax.set_ylabel("UTM Northing (m)")

    ax.set_title(
        os.path.basename(file),
        fontsize=12
    )

    plt.tight_layout()

    # -----------------------------------------------------
    # SAVE FIGURE TO MEMORY
    # -----------------------------------------------------

    buffer = io.BytesIO()

    fig.savefig(
        buffer,
        format="png",
        bbox_inches="tight"
    )

    plt.close(fig)

    buffer.seek(0)

    return buffer.read()


# =========================================================
# SAVE LABELS
# =========================================================

def save_labels():

    df = pd.DataFrame([
        {
            "file": file,
            "label": label
        }
        for file, label in labels.items()
    ])

    df.to_csv(
        LABEL_FILE,
        index=False
    )


# =========================================================
# REMAINING IMAGES
# =========================================================

remaining = [
    str(f)
    for f in files
    if str(f) not in labels
]

print(f"Images remaining: {len(remaining)}")


# =========================================================
# STATE
# =========================================================

current_index = 0


# =========================================================
# WIDGETS
# =========================================================

image_widget = widgets.Image(
    format="png",
    width=900
)

status = widgets.HTML()


# =========================================================
# BUTTONS
# =========================================================

btn_previous = widgets.Button(
    description="← Previous"
)

btn_clear = widgets.Button(
    description="CLEAR",
    button_style="success"
)

btn_partial = widgets.Button(
    description="PARTIAL CLOUD",
    button_style="warning"
)

btn_cloudy = widgets.Button(
    description="CLOUDY",
    button_style="danger"
)

btn_not_in_scene = widgets.Button(
    description="NOT IN SCENE"
)

btn_next = widgets.Button(
    description="Next →"
)


# =========================================================
# DISPLAY CURRENT IMAGE
# =========================================================

def show_image():

    global current_index

    if len(remaining) == 0:

        status.value = "<b>All images have been labelled!</b>"

        image_widget.value = b""

        return

    if current_index < 0:
        current_index = 0

    if current_index >= len(remaining):
        current_index = len(remaining) - 1

    file = remaining[current_index]

    # Create PNG
    png = make_image_png(file)

    # Display
    image_widget.value = png

    status.value = (
        f"<b>Image {current_index + 1} "
        f"of {len(remaining)}</b><br>"
        f"{os.path.basename(file)}"
    )


# =========================================================
# LABEL IMAGE
# =========================================================

def label_image(label):

    global current_index

    file = remaining[current_index]

    # Save label
    labels[file] = label

    # Write immediately
    save_labels()

    print(
        f"{os.path.basename(file)} → {label}"
    )

    # Next image
    current_index += 1

    show_image()


# =========================================================
# BUTTON CALLBACKS
# =========================================================

def clear_clicked(b):
    label_image("clear")


def partial_clicked(b):
    label_image("partial_cloud")


def cloudy_clicked(b):
    label_image("cloudy")


def not_in_scene_clicked(b):
    label_image("not_in_scene")

def previous_clicked(b):

    global current_index

    if current_index > 0:

        current_index -= 1
        show_image()


def next_clicked(b):

    global current_index

    if current_index < len(remaining) - 1:

        current_index += 1
        show_image()


# =========================================================
# CONNECT BUTTONS
# =========================================================

btn_clear.on_click(clear_clicked)
btn_partial.on_click(partial_clicked)
btn_cloudy.on_click(cloudy_clicked)
btn_not_in_scene.on_click(not_in_scene_clicked)

btn_previous.on_click(previous_clicked)
btn_next.on_click(next_clicked)


# =========================================================
# DISPLAY
# =========================================================

display(status)

display(
    widgets.HBox([
        btn_previous,
        btn_clear,
        btn_partial,
        btn_cloudy,
        btn_not_in_scene,
        btn_next
    ])
)

display(image_widget)

show_image()

Already labelled: 0
Images remaining: 206


HTML(value='')

Image(value=b'', width='900')

AST_08_00403082008145510_20250620191042_SKT.tif → clear
AST_08_00403082008145519_20250620191317_SKT.tif → clear
AST_08_00403152008150116_20250620211308_SKT.tif → cloudy
AST_08_00404122003145553_20250308145600_SKT.tif → clear
AST_08_00404122003145602_20250308145051_SKT.tif → clear
AST_08_00404212003144939_20250308192344_SKT.tif → clear
AST_08_00404212018145615_20250826195902_SKT.tif → clear
AST_08_00404212018145623_20250826200305_SKT.tif → clear
AST_08_00405022023143159_20250927152529_SKT.tif → clear
AST_08_00405022023143208_20250927152401_SKT.tif → clear
AST_08_00405032014150146_20250727213331_SKT.tif → cloudy
AST_08_00405062015150202_20250806030852_SKT.tif → cloudy
AST_08_00405062017144313_20250821044952_SKT.tif → clear
AST_08_00405072003144929_20250309105559_SKT.tif → clear
AST_08_00405122014145557_20250728004158_SKT.tif → clear
AST_08_00405192014150205_20250728044838_SKT.tif → cloudy
AST_08_00405232004150132_20250327114514_SKT.tif → not_in_scene
AST_08_00406022015144356_202508062027

AST_08_00408072014150147_20250801044119_SKT.tif → clear
AST_08_00408072020150128_20250910145950_SKT.tif → cloudy
AST_08_00408082002145042_20250225063857_SKT.tif → clear
AST_08_00408082017145545_20250822214514_SKT.tif → cloudy
AST_08_00408082017145554_20250822214813_SKT.tif → cloudy
AST_08_00408102024140707_20251006105632_SKT.tif → cloudy
AST_08_00408102024140715_20251006105842_SKT.tif → cloudy
AST_08_00408112003144827_20250313161236_SKT.tif → cloudy
AST_08_00408112004150101_20250330160633_SKT.tif → cloudy
AST_08_00408112023143453_20250929160437_SKT.tif → clear
AST_08_00408122016150214_20250815094318_SKT.tif → cloudy
AST_08_00408132004144847_20250330175401_SKT.tif → clear
AST_08_00408142005150047_20250520053338_SKT.tif → cloudy
AST_08_00408142016144959_20250815110608_SKT.tif → clear
AST_08_00408152002145651_20250225124200_SKT.tif → cloudy
AST_08_00408152002145700_20250225124455_SKT.tif → cloudy
AST_08_00408152017150158_20250823015501_SKT.tif → not_in_scene
AST_08_00408152024141140_20251